# BSARD Question Analysis by PDF Extraction Status

Categorise each of the 1,108 BSARD questions by the PDF-extraction status of its mapped (ground-truth) articles.

**Unit of analysis: a unique BSARD article, identified by `bsard_id`.**

The corpus database stores 33,741 BSARD article *rows* across 22,633 unique `bsard_id` values — the same logical BSARD article can appear in multiple PDFs (multi-part codes). This notebook collapses to one record per `bsard_id`, which is the same identifier used in the canonical HuggingFace BSARD dataset (`load_dataset('maastrichtlawtech/bsard', 'corpus')`). Downstream projects should join on `bsard_id`.

**Buckets**

| Bucket | Rule |
| --- | --- |
| `exact` | All relevant BSARD articles have `verification_status = 'FOUND'` |
| `partial` | All relevant BSARD articles have `verification_status = 'PARTIAL'` |
| `not_present` | All relevant BSARD articles have `verification_status = 'NOT FOUND'` (HF-only stubs always fall here) |
| `mixed` | Relevant BSARD articles span more than one of the above buckets |

Outputs: `output/question_analysis/questions_by_extraction_status.jsonl` and `summary.json` (gitignored locally; included in the published Hugging Face dataset). See `QUESTION_EXTRACTION_ANALYSIS.md` in the project root.

## 1. Setup

In [ ]:
import json
import sqlite3
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DB_PATH = PROJECT_ROOT / "output" / "bsard_corpus.db"
OUT_DIR = PROJECT_ROOT / "output" / "question_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

JSONL_PATH = OUT_DIR / "questions_by_extraction_status.jsonl"
SUMMARY_PATH = OUT_DIR / "summary.json"

print(f"DB: {DB_PATH}")
print(f"Out: {OUT_DIR}")
assert DB_PATH.exists(), DB_PATH

In [ ]:
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

articles = pd.read_sql(
    """
    SELECT article_id, bsard_id, is_bsard_article, verification_status,
           pdf_filename, law_code, law_type, article_status, is_pre_bsard,
           article_number
    FROM articles
    """,
    conn,
)
questions = pd.read_sql(
    """
    SELECT question_id, split, n_relevant_articles, relevant_article_ids,
           relevant_bsard_ids
    FROM questions
    """,
    conn,
)

questions["relevant_article_ids"] = questions["relevant_article_ids"].apply(json.loads)
questions["relevant_bsard_ids"] = questions["relevant_bsard_ids"].apply(json.loads)

print(f"article rows : {len(articles):,}")
print(f"  BSARD rows : {(articles['is_bsard_article'] == 1).sum():,}")
print(f"  unique bsard_id : {articles['bsard_id'].nunique():,}")
print(f"questions    : {len(questions):,}  (train={(questions['split']=='train').sum()}, test={(questions['split']=='test').sum()})")

## 2. Build a per-`bsard_id` lookup

`verification_status` is consistent across all rows that share a `bsard_id` (verified — 0 inconsistent ids). `pdf_filename` can vary: the same BSARD article may be present in several PDFs of multi-part codes (Code Civil, Code Judiciaire, etc.), so we collect the deduplicated set of filenames per `bsard_id`.

In [ ]:
bsard_rows = articles[articles["is_bsard_article"] == 1].copy()

status_per_bsard = (
    bsard_rows.groupby("bsard_id")["verification_status"].first().to_dict()
)
pdfs_per_bsard = (
    bsard_rows.dropna(subset=["pdf_filename"])
    .groupby("bsard_id")["pdf_filename"]
    .agg(lambda s: sorted(set(s)))
    .to_dict()
)

print(f"unique bsard_ids        : {len(status_per_bsard):,}")
print(f"  with at least one PDF : {len(pdfs_per_bsard):,}")
print(f"  HF-only stubs (no PDF): {len(status_per_bsard) - len(pdfs_per_bsard):,}")

## 3. Categorise questions

Drive directly from `questions.relevant_bsard_ids` (the list of unique BSARD article IDs the question maps to). One question can list the same `bsard_id` more than once — we dedup.

In [ ]:
STATUS_TO_BUCKET = {
    "FOUND": "exact",
    "PARTIAL": "partial",
    "NOT FOUND": "not_present",
}

rows = []
questions_with_dup_bsard = 0
missing_bsard_ids = []

for _, q in questions.iterrows():
    raw_ids = q["relevant_bsard_ids"]
    unique_ids = sorted(set(raw_ids))
    if len(unique_ids) != len(raw_ids):
        questions_with_dup_bsard += 1

    per_article = []
    buckets_seen = set()
    pdfs_seen = set()
    for bid in unique_ids:
        if bid not in status_per_bsard:
            missing_bsard_ids.append((q["question_id"], bid))
            continue
        status = status_per_bsard[bid]
        bucket = STATUS_TO_BUCKET.get(status, "not_present")
        buckets_seen.add(bucket)
        pdfs = pdfs_per_bsard.get(bid, [])
        pdfs_seen.update(pdfs)
        per_article.append({
            "bsard_id": int(bid),
            "verification_status": status,
            "pdf_filenames": pdfs,
        })

    if not buckets_seen:
        bucket = "not_present"
    elif len(buckets_seen) == 1:
        bucket = next(iter(buckets_seen))
    else:
        bucket = "mixed"

    rows.append({
        "question_id": int(q["question_id"]),
        "split": q["split"],
        "extraction_status": bucket,
        "n_relevant_bsard_articles": len(unique_ids),
        "pdf_filenames": sorted(pdfs_seen),
        "bsard_articles": per_article,
    })

result = pd.DataFrame(rows)
print(f"Categorised {len(result):,} questions.")
print(f"Questions with duplicate bsard_id in relevant_bsard_ids: {questions_with_dup_bsard}")
print(f"Unresolved bsard_ids: {len(missing_bsard_ids)}")

In [ ]:
summary = (
    result.groupby(["split", "extraction_status"]).size().unstack(fill_value=0)
)
for col in ["exact", "partial", "not_present", "mixed"]:
    if col not in summary.columns:
        summary[col] = 0
summary = summary[["exact", "partial", "not_present", "mixed"]]
summary["total"] = summary.sum(axis=1)
summary.loc["all"] = summary.sum(axis=0)
summary

In [ ]:
totals = result.groupby("extraction_status").size().reindex(
    ["exact", "partial", "not_present", "mixed"], fill_value=0
)
pct = (totals / totals.sum() * 100).round(2)
pd.DataFrame({"count": totals, "pct": pct})

## 4. Write outputs

JSONL chosen because each row carries a nested per-article list. Format matches the project's existing exports.

In [ ]:
with JSONL_PATH.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

summary_payload = {
    "unit_of_analysis": "bsard_id (unique BSARD article id, matching `id` field in maastrichtlawtech/bsard corpus split)",
    "total_questions": int(len(result)),
    "buckets": [
        {"name": name, "count": int(totals[name]), "pct": float(pct[name])}
        for name in ["exact", "partial", "not_present", "mixed"]
    ],
    "by_split": {
        split: result[result["split"] == split]["extraction_status"].value_counts().to_dict()
        for split in ["train", "test"]
    },
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Wrote {JSONL_PATH}")
print(f"Wrote {SUMMARY_PATH}")

## 5. Lookup cell

Set `QUESTION_ID` and re-run to inspect a single question end-to-end. Article text is fetched once per `bsard_id` from any PDF row that carries it (canonical text comes from the BSARD HF dataset for BSARD articles, so all rows for one `bsard_id` agree).

In [ ]:
QUESTION_ID = 1  # change me

q_text = pd.read_sql(
    "SELECT question_text FROM questions WHERE question_id = ?",
    conn, params=(QUESTION_ID,),
)["question_text"].iloc[0]

row = next(r for r in rows if r["question_id"] == QUESTION_ID)

print(f"=== Question {QUESTION_ID} ({row['split']}) ===")
print(q_text)
print()
print(f"extraction_status         : {row['extraction_status']}")
print(f"n_relevant_bsard_articles : {row['n_relevant_bsard_articles']}")
print(f"pdf_filenames             : {row['pdf_filenames']}")
print()
for i, a in enumerate(row["bsard_articles"], 1):
    text = pd.read_sql(
        """
        SELECT law_code, article_number, article_text
        FROM articles
        WHERE bsard_id = ?
        LIMIT 1
        """,
        conn, params=(a["bsard_id"],),
    ).iloc[0]
    print(f"--- [{i}/{len(row['bsard_articles'])}] bsard_id={a['bsard_id']} ---")
    print(f"    {text['law_code']} -- Art. {text['article_number']}")
    print(f"    verification_status : {a['verification_status']}")
    print(f"    pdf_filenames       : {a['pdf_filenames']}")
    body = text["article_text"] or ""
    snippet = body if len(body) <= 600 else body[:600] + "..."
    print(f"    text                : {snippet}")
    print()

## 6. Filter cells

Each cell takes a `FILTERS` dict at the top. Set the keys you want to constrain on (use `None` to skip a filter) and re-run.

### 6.1 Filter questions

In [ ]:
FILTERS = {
    "split": None,                      # 'train' | 'test' | None
    "extraction_status": None,          # 'exact' | 'partial' | 'not_present' | 'mixed' | None
    "n_relevant_min": None,             # int | None  (inclusive, on n_relevant_bsard_articles)
    "n_relevant_max": None,             # int | None  (inclusive, on n_relevant_bsard_articles)
}

df = result.copy()
if FILTERS["split"] is not None:
    df = df[df["split"] == FILTERS["split"]]
if FILTERS["extraction_status"] is not None:
    df = df[df["extraction_status"] == FILTERS["extraction_status"]]
if FILTERS["n_relevant_min"] is not None:
    df = df[df["n_relevant_bsard_articles"] >= FILTERS["n_relevant_min"]]
if FILTERS["n_relevant_max"] is not None:
    df = df[df["n_relevant_bsard_articles"] <= FILTERS["n_relevant_max"]]

print(f"{len(df):,} questions match.")
df[["question_id", "split", "extraction_status", "n_relevant_bsard_articles"]].head(20)

### 6.2 Filter BSARD articles

Operates on the deduplicated BSARD article view (one row per `bsard_id`). Joins back to all PDF appearances.

In [ ]:
FILTERS = {
    "law_code": None,             # exact match | None
    "verification_status": None,  # 'FOUND' | 'PARTIAL' | 'NOT FOUND' | None
    "pdf_filename": None,         # exact match against any of the article's PDFs | None
    "is_pre_bsard": None,         # 0 | 1 | None
    "article_status": None,       # 'ORIGINAL_NEVER_AMENDED' | 'PRE_BSARD' | 'POST_BSARD' | 'PARSE_ERROR' | None
}

first_row = bsard_rows.groupby("bsard_id").first().reset_index()
first_row["pdf_filenames"] = first_row["bsard_id"].map(
    lambda b: pdfs_per_bsard.get(b, [])
)
df = first_row.copy()

if FILTERS["law_code"] is not None:
    df = df[df["law_code"] == FILTERS["law_code"]]
if FILTERS["verification_status"] is not None:
    df = df[df["verification_status"] == FILTERS["verification_status"]]
if FILTERS["pdf_filename"] is not None:
    df = df[df["pdf_filenames"].apply(lambda l: FILTERS["pdf_filename"] in l)]
if FILTERS["is_pre_bsard"] is not None:
    df = df[df["is_pre_bsard"] == FILTERS["is_pre_bsard"]]
if FILTERS["article_status"] is not None:
    df = df[df["article_status"] == FILTERS["article_status"]]

print(f"{len(df):,} unique BSARD articles match.")
df[[
    "bsard_id", "law_code", "article_number", "verification_status",
    "pdf_filenames", "is_pre_bsard", "article_status",
]].head(20)

### 6.3 PDF aggregate filter

For each PDF: how many questions point into it (counted once per question via `pdf_filenames`), and how many *unique* BSARD articles it contains broken down by `verification_status`.

In [ ]:
FILTERS = {
    "min_questions": None,   # int | None
    "max_questions": None,   # int | None
    "contains": None,        # substring of pdf_filename | None
}

q_per_pdf = Counter()
for r in rows:
    for pdf in r["pdf_filenames"]:
        q_per_pdf[pdf] += 1

exploded = (
    bsard_rows.dropna(subset=["pdf_filename"])[["bsard_id", "pdf_filename", "verification_status"]]
    .drop_duplicates(["bsard_id", "pdf_filename"])
)
art_per_pdf_status = (
    exploded.groupby(["pdf_filename", "verification_status"]).size().unstack(fill_value=0)
)
for col in ["FOUND", "PARTIAL", "NOT FOUND"]:
    if col not in art_per_pdf_status.columns:
        art_per_pdf_status[col] = 0

pdf_summary = art_per_pdf_status[["FOUND", "PARTIAL", "NOT FOUND"]].copy()
pdf_summary["n_questions"] = pdf_summary.index.map(lambda p: q_per_pdf.get(p, 0))
pdf_summary = pdf_summary.reset_index().rename_axis(None, axis=1)

df = pdf_summary
if FILTERS["min_questions"] is not None:
    df = df[df["n_questions"] >= FILTERS["min_questions"]]
if FILTERS["max_questions"] is not None:
    df = df[df["n_questions"] <= FILTERS["max_questions"]]
if FILTERS["contains"] is not None:
    df = df[df["pdf_filename"].str.contains(FILTERS["contains"], na=False)]

df = df.sort_values("n_questions", ascending=False)
print(f"{len(df):,} PDFs match.")
df.head(20)